In [0]:
gold_df = spark.table("gold_interaction_matrix")

display(gold_df.limit(5))

user_id,product_id,rating
175918,10761,1
147650,25533,3
189157,23543,2
189157,23400,1
47358,27730,7


In [0]:
bronze_products = spark.table("bronze_products")

In [0]:
silver_df = spark.table("silver_customer_products")

In [0]:
from pyspark.sql.functions import count, log1p

gold_df = (
    silver_df
    .groupBy("user_id","product_id")
    .agg(count("*").alias("purchase_count"))
    .withColumn("rating", log1p("purchase_count"))
)

In [0]:
train_data, test_data = gold_df.randomSplit([0.8, 0.2], seed=42)

print(train_data.count())
print(test_data.count())

10646996
2660957


In [0]:
from pyspark.ml.recommendation import ALS
als = ALS(
    userCol="user_id",
    itemCol="product_id",
    ratingCol="rating",

    implicitPrefs=True,

    rank=30,
    maxIter=15,
    regParam=0.05,

    coldStartStrategy="drop"
)

In [0]:
als_model = als.fit(train_data)

In [0]:
predictions = als_model.transform(test_data)

display(predictions.limit(10))

user_id,product_id,purchase_count,rating,prediction
15,30292,1,0.6931471805599453,0.018817298
15,48142,3,1.3862943611198906,0.078699276
15,12427,10,2.3978952727983707,0.206916
15,11266,10,2.3978952727983707,0.16866183
15,14715,11,2.4849066497880004,0.14644216
24,33081,3,1.3862943611198906,0.08802261
29,1261,2,1.0986122886681096,0.003580845
29,8523,2,1.0986122886681096,0.016715763
29,8806,3,1.3862943611198906,0.0012744146
29,33945,2,1.0986122886681096,0.00454343


In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions)

print("RMSE =", rmse)

RMSE = 1.0247089158918026


MODEL-1

In [0]:
als2 = ALS(
    userCol="user_id",
    itemCol="product_id",
    ratingCol="rating",

    implicitPrefs=False,

    rank=30,
    maxIter=15,
    regParam=0.05,

    coldStartStrategy="drop"
)

In [0]:
alsmodel = als2.fit(train_data)

In [0]:
predictions2 = alsmodel.transform(test_data)

display(predictions2.limit(10))

user_id,product_id,purchase_count,rating,prediction
15,30292,1,0.6931471805599453,0.88191366
15,48142,3,1.3862943611198906,1.4827309
15,12427,10,2.3978952727983707,2.2772634
15,11266,10,2.3978952727983707,2.1197472
15,14715,11,2.4849066497880004,2.1905923
24,33081,3,1.3862943611198906,1.1662723
29,1261,2,1.0986122886681096,0.96852934
29,8523,2,1.0986122886681096,1.1390107
29,8806,3,1.3862943611198906,1.080264
29,33945,2,1.0986122886681096,1.1736205


In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions2)

print("RMSE =", rmse)

RMSE = 0.40605408659912523


## MODEL-2

In [0]:
als3 = ALS(
    userCol="user_id",
    itemCol="product_id",
    ratingCol="rating",

    implicitPrefs=False,

    rank = 50,
    maxIter = 20,
    regParam = 0.01,

    coldStartStrategy="drop"
)

In [0]:
als__model = als3.fit(train_data)

In [0]:
predictions3 = als__model.transform(test_data)

display(predictions3.limit(10))

user_id,product_id,purchase_count,rating,prediction
15,30292,1,0.6931471805599453,0.997513
15,48142,3,1.3862943611198906,1.3941295
15,12427,10,2.3978952727983707,2.3702745
15,11266,10,2.3978952727983707,2.3624756
15,14715,11,2.4849066497880004,2.44016
24,33081,3,1.3862943611198906,1.3365258
29,1261,2,1.0986122886681096,1.1078686
29,8523,2,1.0986122886681096,1.3649856
29,8806,3,1.3862943611198906,1.333103
29,33945,2,1.0986122886681096,1.1099427


In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions3)

print("RMSE =", rmse)

RMSE = 0.32208276900397365


## Generate Recommendations

In [0]:
als__model.recommendForAllUsers(5)

In [0]:
print(type(als__model))

<class 'pyspark.ml.recommendation.ALSModel'>


In [0]:
als__model.userFactors.show(5)

als__model.itemFactors.show(5)

+---+--------------------+
| id|            features|
+---+--------------------+
| 10|[0.3866328, -0.14...|
| 20|[0.1439986, -0.22...|
| 30|[0.15019374, 0.04...|
| 40|[0.8817575, -0.11...|
| 50|[-0.07701446, -0....|
+---+--------------------+
only showing top 5 rows
+---+--------------------+
| id|            features|
+---+--------------------+
| 10|[0.19972208, 0.27...|
| 20|[0.078698866, -0....|
| 30|[0.18613352, -0.0...|
| 40|[0.2051544, -0.30...|
| 50|[-0.06244727, 0.0...|
+---+--------------------+
only showing top 5 rows


In [0]:
gold_df.select("user_id").distinct().show(20)

+-------+
|user_id|
+-------+
|   2476|
| 204424|
|  28386|
| 177387|
| 203684|
| 137784|
|    485|
| 151277|
| 112853|
|  50334|
| 205622|
|  29705|
|  76423|
|  74395|
| 186053|
|  59211|
|  51993|
| 161897|
|  88002|
|  62512|
+-------+
only showing top 20 rows


In [0]:
user_id= 53003

In [0]:
purchased = (

    silver_df

    .filter(silver_df.user_id==user_id)

    .select("product_id","product_name")
    .distinct()

)

display(purchased)

product_id,product_name
29616,Direct Trade Black Cat Classic Espresso Roast Whole Bean Coffee
27299,Organic SprouTofu Firm Water Packed Tofu
7041,Pecorino Romano
43154,Sparkling Mineral Water
35168,Ezekiel 4:9 Sprouted Grain Tortillas
27521,Organic Lacinato (Dinosaur) Kale
18531,Organic Heavy Whipping Cream
46667,Organic Ginger Root
16882,Baby Spring Mix
41950,Organic Tomato Cluster


In [0]:
products_df = spark.table("bronze_products")

products_df.show(5)

+----------+--------------------+--------+-------------+
|product_id|        product_name|aisle_id|department_id|
+----------+--------------------+--------+-------------+
|         1|Chocolate Sandwic...|      61|           19|
|         2|    All-Seasons Salt|     104|           13|
|         3|Robust Golden Uns...|      94|            7|
|         4|Smart Ones Classi...|      38|            1|
|         5|Green Chile Anyti...|       5|           13|
+----------+--------------------+--------+-------------+
only showing top 5 rows


In [0]:
candidate_products = products_df.select("product_id")

candidate_products.show(5)

+----------+
|product_id|
+----------+
|         1|
|         2|
|         3|
|         4|
|         5|
+----------+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import lit

user_products = (
    candidate_products
    .withColumn("user_id", lit(user_id))
)

display(user_products.limit(5))

product_id,user_id
1,53003
2,53003
3,53003
4,53003
5,53003


In [0]:
pred = als__model.transform(user_products)

display(pred.limit(10))

product_id,user_id,prediction
1,53003,1.1931807
2,53003,0.69691265
3,53003,1.6495295
4,53003,0.7715403
5,53003,0.90924996
6,53003,1.0513927
7,53003,0.83482975
8,53003,0.6380334
9,53003,1.308371
10,53003,1.0075295


In [0]:
recommended = (
    pred.join(
        purchased.select("product_id"),
        on="product_id",
        how="left_anti"
    )
)

display(recommended.limit(5))

product_id,user_id,prediction
1,53003,1.1931807
2,53003,0.69691265
3,53003,1.6495295
4,53003,0.7715403
5,53003,0.90924996


In [0]:
top5 = (
    recommended
    .orderBy(
        recommended.prediction.desc()
    )
    .limit(5)
)

display(top5)

product_id,user_id,prediction
13128,53003,3.250427
14466,53003,3.2084148
3497,53003,2.9547203
1406,53003,2.8975515
47231,53003,2.8091109


In [0]:
final_rec = (
    top5
    .join(products_df, on="product_id")
)

display(final_rec)

product_id,user_id,prediction,product_name,aisle_id,department_id
1406,53003,2.8975515,Vegan Nutritional Shake Sweet Vanilla Bean,65,11
3497,53003,2.9547203,"Grape Wine, Delicious Red",28,5
13128,53003,3.250427,Purified Alkalkine Water with Minerals pH10,115,7
14466,53003,3.2084148,Oat Chocolate Chip Coconut Whenever Bars,3,19
47231,53003,2.8091109,Ultra-Purified Water,115,7


In [0]:
als__model.save(
"/Volumes/workspace/default/e-commerce_recommendation/best_als_model"
)

---------------------------------------------------------------------------
UnknownException                          Traceback (most recent call last)
File <command-8411869437893501>, line 1
----> 1 als__model.save(
      2 "/Volumes/workspace/default/e-commerce_recommendation/best_als_model"
      3 )

File /databricks/python/lib/python3.12/site-packages/pyspark/ml/util.py:820, in MLWritable.save(self, path)
    818 def save(self, path: str) -> None:
    819     """Save this ML instance to the given path, a shortcut of 'write().save(path)'."""
--> 820     self.write().save(path)

File /databricks/python/lib/python3.12/site-packages/pyspark/ml/connect/readwrite.py:52, in RemoteMLWriter.save(self, path)
     49 session = SparkSession.getActiveSession()
     50 assert session is not None
---> 52 RemoteMLWriter.saveInstance(
     53     self._instance,
     54     path,
     55     session,
     56     self.shouldOverwrite,
     57     self.optionMap,
     58 )

File /databricks/python/l

In [0]:
top5.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- prediction: float (nullable = false)



In [0]:
top5_named = (
    top5.join(
        bronze_products,
        on="product_id",
        how="left"
    )
)

display(top5_named)

product_id,user_id,prediction,product_name,aisle_id,department_id
13128,53003,3.250427,Purified Alkalkine Water with Minerals pH10,115,7
14466,53003,3.2084148,Oat Chocolate Chip Coconut Whenever Bars,3,19
3497,53003,2.9547203,"Grape Wine, Delicious Red",28,5
1406,53003,2.8975515,Vegan Nutritional Shake Sweet Vanilla Bean,65,11
47231,53003,2.8091109,Ultra-Purified Water,115,7


In [0]:
top5_named.write.mode("overwrite").saveAsTable("recommended_products")

In [0]:
top5_named = (
    top5.join(
        bronze_products,
        on="product_id",
        how="left"
    )
)

display(top5_named)

product_id,user_id,prediction,product_name,aisle_id,department_id
13128,53003,3.250427,Purified Alkalkine Water with Minerals pH10,115,7
14466,53003,3.2084148,Oat Chocolate Chip Coconut Whenever Bars,3,19
3497,53003,2.9547203,"Grape Wine, Delicious Red",28,5
1406,53003,2.8975515,Vegan Nutritional Shake Sweet Vanilla Bean,65,11
47231,53003,2.8091109,Ultra-Purified Water,115,7


In [0]:
spark.sql("SHOW TABLES").filter("tableName='recommended_products'").show()

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
| default|recommended_products|      false|
+--------+--------------------+-----------+



In [0]:
pred.show()

+----------+-------+----------+
|product_id|user_id|prediction|
+----------+-------+----------+
|         1|  53003| 1.1931807|
|         2|  53003|0.69691265|
|         3|  53003| 1.6495295|
|         4|  53003| 0.7715403|
|         5|  53003|0.90924996|
|         6|  53003| 1.0513927|
|         7|  53003|0.83482975|
|         8|  53003| 0.6380334|
|         9|  53003|  1.308371|
|        10|  53003| 1.0075295|
|        11|  53003| 1.3336849|
|        12|  53003|0.68727297|
|        13|  53003| 0.5424413|
|        14|  53003|0.61082697|
|        15|  53003|0.47327843|
|        16|  53003| 0.7849888|
|        17|  53003|0.97678494|
|        18|  53003| 1.2532743|
|        19|  53003| 0.8817252|
|        20|  53003| 0.8202603|
+----------+-------+----------+
only showing top 20 rows


In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

mae_evaluator = RegressionEvaluator(
    metricName="mae",
    labelCol="rating",
    predictionCol="prediction"
)

mae = mae_evaluator.evaluate(predictions3)

print("MAE =", mae)

MAE = 0.1954217845448414


In [0]:
# Generate Top-10 recommendations for every user
top10_recommendations = als_model.recommendForAllUsers(10)

display(top10_recommendations.limit(5))

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6368237541827930>, line 4
      1 # Generate Top-10 recommendations for every user
      2 top10_recommendations = als_model.recommendForAllUsers(10)
----> 4 display(top10_recommendations.limit(5))

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     self.display_connect_table(input, **kwargs)
    137 elif isinstance(input, ConnectDataFrame):
    138     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:96, in Display.display_connect_table(self, df, **kwargs)
     91 except Exception as e:
     92     raise type(
     93         e
     94     )("IPython shell encountered an error or was missing data, please restart the notebook or

In [0]:
predictions3.printSchema()

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6368237541827931>, line 1
----> 1 predictions3.printSchema()

NameError: name 'predictions3' is not defined